# The scene in the tokens

Fig. 8c and Appendix D.4 and D.5, read with the frozen readout of each checkpoint, without training anything:

- **Geometry of $\mathbf{z}_0$:** the 3D points recovered from $\mathbf{z}_0$ on the 416 held-out ScanNet image sets.
- **Transformation of $\mathbf{z}_k$:** how often a re-emitted $\mathbf{z}_k$ after a camera's viewpoint decodes
  nearest to $\mathbf{z}_0$'s scene seen from that camera (*transformed*), and nearest to the true scene seen from it.

**Needs:** the geometry supervision (`scripts/data/geometry.sh`: `data/smind/fusion`) and the perspective-stage rows with their 300 held-out
rows (`scripts/data/sft.sh`); one GPU, about five minutes per model.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import smind_nb as nb
nb.setup()

SIZE = "4b"   # or "9b"
GPU = "0"

In [ ]:
nb.need("data/smind/fusion/maps_scannet_dev.parquet", "data/smind/sft/gpt-6-astra/perspective/heldout_z.parquet")
res = {}
for stage in ("sft", "rl"):
    out = nb.RUNS / f"{SIZE}-{stage}/transformed_readout.json"
    nb.run(["python", "analysis/method/transformed_readout.py", "--ckpt", nb.checkpoint(SIZE, stage), "--out", out],
           out.with_suffix(".log"), done=out, gpu=GPU)
    r = nb.read_json(out)
    res[stage] = {"ScanNet points recovered from z0": r["z0_point_far_pct"]["maps_scannet_dev"],
                  "z_k transformed (z0's scene)": r["transformed_pct"],
                  "z_k transformed (true scene)": r["transformed_true_pct"]}
import pandas as pd
df = pd.DataFrame(res)
if SIZE == "4b":
    df["paper: sft"] = [76.13, 97.49, 96.67]
    df["paper: rl"] = [75.95, 97.29, None]
display(df)

After SFT the tokens still hold the geometry of pre-training, and nearly every $\mathbf{z}_k$ is the scene re-expressed
at the declared camera. RL, which optimizes only the answer's reward, leaves both in place.